# Bio-PM IRB Analysis (Standard vs Adv)
Single regression notebook for Member 1 pipeline analysis.
Main task: LOSO regression predicting continuous ARAT/FMA from Bio-PM embeddings.


In [ ]:
import sys, os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from collections import defaultdict
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.decomposition import PCA
import umap
warnings.filterwarnings('ignore')
%matplotlib inline

BIOPM_ROOT = "CS690TR"
sys.path.insert(0, BIOPM_ROOT)

STD_LEGACY = "features/biopm_features_legacy_schema.npz"
ADV_LEGACY = "features/biopm_features_legacy_schema_adv.npz"
ADV_WIN    = "features/biopm_features_adv.npz"
RESULTS    = "results"
os.makedirs(f"{RESULTS}/figures", exist_ok=True)
os.makedirs(f"{RESULTS}/metrics", exist_ok=True)
os.makedirs(f"{RESULTS}/splits", exist_ok=True)


## 1) Dataset Overview


In [ ]:
clin = np.load('data/clinical_scores.npz', allow_pickle=True)['clinical_scores'].item()
rows=[]
for (s,w),rec in clin.items():
    rows.append({'subject':int(s),'week':int(w),'ARAT':float(rec.ARAT),'FMA':float(rec.FMA)})
clin_df=pd.DataFrame(rows)
print('Visits:',len(clin_df))
print('Subjects:',clin_df.subject.nunique())
print('ARAT range:',(clin_df.ARAT.min(),clin_df.ARAT.max()))
print('FMA range :',(clin_df.FMA.min(),clin_df.FMA.max()))
healthy=(clin_df.ARAT==57)&(clin_df.FMA==66)
print('Healthy visits convention (ARAT=57,FMA=66):',int(healthy.sum()))
print('Healthy subjects:',clin_df.loc[healthy,'subject'].nunique())
print('Spearman ARAT~FMA:',spearmanr(clin_df.ARAT,clin_df.FMA))

fig,ax=plt.subplots(1,2,figsize=(10,4))
sns.histplot(clin_df.ARAT,bins=20,ax=ax[0],kde=False)
ax[0].set_title('ARAT distribution')
sns.histplot(clin_df.FMA,bins=20,ax=ax[1],kde=False)
ax[1].set_title('FMA distribution')
plt.tight_layout()
plt.show()


## 2) Load Features (Standard + Adv)


In [ ]:
def load_legacy(path):
    d = np.load(path, allow_pickle=True)
    return {
        'X': np.ascontiguousarray(d['features'], dtype=np.float32),
        'Xe': np.ascontiguousarray(d['features_even'], dtype=np.float32),
        'Xo': np.ascontiguousarray(d['features_odd'], dtype=np.float32),
        'arat': d['arat'].astype(float),
        'fma': d['fma'].astype(float),
        'pids': d['pids'].astype(int),
        'labels': d['labels'].astype(int),
        'subjects': d['subjects'].astype(int),
        'weeks': d['weeks'].astype(int),
    }

def quality(name, X):
    print(f"{name}: shape={X.shape}, NaN={np.isnan(X).sum()}, zero_rows={(np.abs(X)<1e-8).all(axis=1).sum()}")
    print(f"  acc mean [0:64]  abs={np.abs(X[:,0:64]).mean():.5f}")
    print(f"  acc std  [64:128] abs={np.abs(X[:,64:128]).mean():.5f}")
    print(f"  grav     [128:]   abs={np.abs(X[:,128:]).mean():.5f}")

std = load_legacy(STD_LEGACY)
adv = load_legacy(ADV_LEGACY)
quality('Standard', std['X'])
quality('Adv', adv['X'])


## 3) LOSO Regression Helpers


In [ ]:
def loso_regression(X, y_scores, pids, alpha=1.0, label=''):
    logo = LeaveOneGroupOut()
    y_true, y_pred = [], []
    for tr, te in logo.split(X, y_scores, groups=pids):
        X_tr, y_tr = X[tr], y_scores[tr]
        X_te, y_te = X[te], y_scores[te]
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)
        reg = Ridge(alpha=alpha)
        reg.fit(X_tr_s, y_tr)
        y_pred.extend(reg.predict(X_te_s).tolist())
        y_true.extend(y_te.tolist())
    y_true=np.array(y_true); y_pred=np.array(y_pred)
    ss_res=((y_true-y_pred)**2).sum(); ss_tot=((y_true-y_true.mean())**2).sum()
    r2=1-ss_res/ss_tot
    rmse=np.sqrt(((y_true-y_pred)**2).mean())
    sr,sp=spearmanr(y_true,y_pred)
    print(f"{label}: R²={r2:.3f} RMSE={rmse:.2f} Spearman ρ={sr:.3f} (p={sp:.3g})")
    return {'r2':r2,'rmse':rmse,'spearman_r':sr,'spearman_p':sp,'y_true':y_true,'y_pred':y_pred}

def loso_regression_pca(X, y_scores, pids, n_components=50, alpha=1.0, label=''):
    logo = LeaveOneGroupOut()
    y_true, y_pred = [], []
    for tr, te in logo.split(X, y_scores, groups=pids):
        X_tr, y_tr = X[tr], y_scores[tr]
        X_te, y_te = X[te], y_scores[te]
        sc=StandardScaler(); X_tr_s=sc.fit_transform(X_tr); X_te_s=sc.transform(X_te)
        pca=PCA(n_components=min(n_components, X_tr_s.shape[0]-1, X_tr_s.shape[1]))
        X_tr_p=pca.fit_transform(X_tr_s); X_te_p=pca.transform(X_te_s)
        reg=Ridge(alpha=alpha); reg.fit(X_tr_p,y_tr)
        y_pred.extend(reg.predict(X_te_p).tolist()); y_true.extend(y_te.tolist())
    y_true=np.array(y_true); y_pred=np.array(y_pred)
    ss_res=((y_true-y_pred)**2).sum(); ss_tot=((y_true-y_true.mean())**2).sum()
    r2=1-ss_res/ss_tot; rmse=np.sqrt(((y_true-y_pred)**2).mean()); sr,sp=spearmanr(y_true,y_pred)
    print(f"{label} [PCA]: R²={r2:.3f} RMSE={rmse:.2f} Spearman ρ={sr:.3f} (p={sp:.3g})")
    return {'r2':r2,'rmse':rmse,'spearman_r':sr,'spearman_p':sp,'y_true':y_true,'y_pred':y_pred}

def plot_pred(ax, y_true, y_pred, title):
    ax.scatter(y_true,y_pred,s=30,alpha=0.8)
    lo=min(y_true.min(),y_pred.min()); hi=max(y_true.max(),y_pred.max())
    ax.plot([lo,hi],[lo,hi],'r--',lw=1)
    ax.set_title(title); ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')


## 4) Standard Pipeline Regression


In [ ]:
res_std_arat = loso_regression(std['X'], std['arat'], std['pids'], label='Standard -> ARAT')
res_std_fma  = loso_regression(std['X'], std['fma'],  std['pids'], label='Standard -> FMA')
res_std_arat_pca = loso_regression_pca(std['X'], std['arat'], std['pids'], label='Standard -> ARAT')
res_std_fma_pca  = loso_regression_pca(std['X'], std['fma'],  std['pids'], label='Standard -> FMA')

fig,ax=plt.subplots(1,2,figsize=(10,4))
plot_pred(ax[0],res_std_arat['y_true'],res_std_arat['y_pred'],'Standard ARAT')
plot_pred(ax[1],res_std_fma['y_true'],res_std_fma['y_pred'],'Standard FMA')
plt.tight_layout(); plt.savefig('results/figures/regression_standard.png',dpi=150); plt.show()

pd.DataFrame([
    {'target':'ARAT','pipeline':'Standard','r2':res_std_arat['r2'],'rmse':res_std_arat['rmse'],'spearman_r':res_std_arat['spearman_r']},
    {'target':'FMA','pipeline':'Standard','r2':res_std_fma['r2'],'rmse':res_std_fma['rmse'],'spearman_r':res_std_fma['spearman_r']},
]).to_csv('results/metrics/regression_standard.csv',index=False)


## 5) Adv Pipeline Regression


In [ ]:
res_adv_arat = loso_regression(adv['X'], adv['arat'], adv['pids'], label='Adv -> ARAT')
res_adv_fma  = loso_regression(adv['X'], adv['fma'],  adv['pids'], label='Adv -> FMA')
res_adv_arat_pca = loso_regression_pca(adv['X'], adv['arat'], adv['pids'], label='Adv -> ARAT')
res_adv_fma_pca  = loso_regression_pca(adv['X'], adv['fma'],  adv['pids'], label='Adv -> FMA')

fig,ax=plt.subplots(1,2,figsize=(10,4))
plot_pred(ax[0],res_adv_arat['y_true'],res_adv_arat['y_pred'],'Adv ARAT')
plot_pred(ax[1],res_adv_fma['y_true'],res_adv_fma['y_pred'],'Adv FMA')
plt.tight_layout(); plt.savefig('results/figures/regression_adv.png',dpi=150); plt.show()

pd.DataFrame([
    {'target':'ARAT','pipeline':'Adv','r2':res_adv_arat['r2'],'rmse':res_adv_arat['rmse'],'spearman_r':res_adv_arat['spearman_r']},
    {'target':'FMA','pipeline':'Adv','r2':res_adv_fma['r2'],'rmse':res_adv_fma['rmse'],'spearman_r':res_adv_fma['spearman_r']},
]).to_csv('results/metrics/regression_adv.csv',index=False)


## 6) Standard vs Adv Comparison


In [ ]:
comp = pd.DataFrame([
    {'target':'ARAT','pipeline':'Standard','r2':res_std_arat['r2'],'rmse':res_std_arat['rmse'],'spearman_r':res_std_arat['spearman_r']},
    {'target':'ARAT','pipeline':'Adv','r2':res_adv_arat['r2'],'rmse':res_adv_arat['rmse'],'spearman_r':res_adv_arat['spearman_r']},
    {'target':'FMA','pipeline':'Standard','r2':res_std_fma['r2'],'rmse':res_std_fma['rmse'],'spearman_r':res_std_fma['spearman_r']},
    {'target':'FMA','pipeline':'Adv','r2':res_adv_fma['r2'],'rmse':res_adv_fma['rmse'],'spearman_r':res_adv_fma['spearman_r']},
])
comp.to_csv('results/metrics/regression_comparison.csv',index=False)
display(comp)

fig,ax=plt.subplots(2,2,figsize=(10,8))
plot_pred(ax[0,0],res_std_arat['y_true'],res_std_arat['y_pred'],'ARAT Standard')
plot_pred(ax[0,1],res_adv_arat['y_true'],res_adv_arat['y_pred'],'ARAT Adv')
plot_pred(ax[1,0],res_std_fma['y_true'],res_std_fma['y_pred'],'FMA Standard')
plot_pred(ax[1,1],res_adv_fma['y_true'],res_adv_fma['y_pred'],'FMA Adv')
plt.tight_layout(); plt.savefig('results/figures/regression_comparison.png',dpi=150); plt.show()


## 7) Aggregation Experiments (Adv)


In [ ]:
dim_groups={
    'full_1028':(0,1028),
    'acc_128':(0,128),
    'acc_meanpool_64':(0,64),
    'acc_stdpool_64':(64,128),
    'gravity_900':(128,1028),
}
rows=[]
for name,(a,b) in dim_groups.items():
    rA=loso_regression(adv['X'][:,a:b],adv['arat'],adv['pids'],label=f'{name} -> ARAT')
    rF=loso_regression(adv['X'][:,a:b],adv['fma'],adv['pids'],label=f'{name} -> FMA')
    rows += [
        {'experiment':'dim_group','mode':name,'target':'ARAT','r2':rA['r2'],'rmse':rA['rmse'],'spearman_r':rA['spearman_r']},
        {'experiment':'dim_group','mode':name,'target':'FMA','r2':rF['r2'],'rmse':rF['rmse'],'spearman_r':rF['spearman_r']},
    ]

adv_win=np.load(ADV_WIN,allow_pickle=True)
Xw=adv_win['features'].astype(np.float32); sw=adv_win['subj'].astype(int); ww=adv_win['week'].astype(int)

def aggregate_windows(features_win, subj, week, mode='mean'):
    groups=defaultdict(list)
    for i,(s,w) in enumerate(zip(subj,week)):
        groups[(s,w)].append(i)
    out=[]
    for key in sorted(groups.keys()):
        idx=np.array(groups[key]); blk=features_win[idx]
        if mode=='mean': out.append(blk.mean(0))
        elif mode=='max': out.append(blk.max(0))
        elif mode=='mean_std_max': out.append(np.concatenate([blk.mean(0),blk.std(0),blk.max(0)]))
    return np.array(out)

# align targets by (subject,week)
key_to_idx={(int(s),int(w)):i for i,(s,w) in enumerate(zip(adv['subjects'],adv['weeks']))}
visit_keys=sorted(set(zip(sw,ww)))
y_arat=np.array([adv['arat'][key_to_idx[k]] for k in visit_keys])
y_fma=np.array([adv['fma'][key_to_idx[k]] for k in visit_keys])
y_pids=np.array([k[0] for k in visit_keys])

for mode in ['mean','max','mean_std_max']:
    Xa=aggregate_windows(Xw,sw,ww,mode=mode)
    rA=loso_regression(Xa,y_arat,y_pids,label=f'agg={mode} -> ARAT')
    rF=loso_regression(Xa,y_fma,y_pids,label=f'agg={mode} -> FMA')
    rows += [
        {'experiment':'window_agg','mode':mode,'target':'ARAT','r2':rA['r2'],'rmse':rA['rmse'],'spearman_r':rA['spearman_r']},
        {'experiment':'window_agg','mode':mode,'target':'FMA','r2':rF['r2'],'rmse':rF['rmse'],'spearman_r':rF['spearman_r']},
    ]

agg_df=pd.DataFrame(rows)
agg_df.to_csv('results/metrics/aggregation_ablation.csv',index=False)
display(agg_df.head(20))

fig,ax=plt.subplots(figsize=(10,4))
plot_df=agg_df[(agg_df['experiment']=='window_agg') & (agg_df['target']=='ARAT')]
sns.barplot(data=plot_df,x='mode',y='r2',ax=ax)
ax.set_title('Adv aggregation ablation (ARAT R²)')
plt.tight_layout(); plt.savefig('results/figures/aggregation_ablation.png',dpi=150); plt.show()


## 8) Spearman Correlation Profiles


In [ ]:
def spearman_profile(X, arat, fma):
    sA=np.zeros(X.shape[1]); sF=np.zeros(X.shape[1])
    for d in range(X.shape[1]):
        sA[d],_=spearmanr(X[:,d],arat)
        sF[d],_=spearmanr(X[:,d],fma)
    return sA,sF

sA_adv,sF_adv=spearman_profile(adv['X'],adv['arat'],adv['fma'])
sA_std,sF_std=spearman_profile(std['X'],std['arat'],std['fma'])

pd.DataFrame({'dim':np.arange(1028),'rho_arat':sA_adv,'rho_fma':sF_adv}).to_csv('results/metrics/spearman_adv.csv',index=False)
pd.DataFrame({'dim':np.arange(1028),'rho_arat':sA_std,'rho_fma':sF_std}).to_csv('results/metrics/spearman_standard.csv',index=False)

def topk(rho,k=10):
    idx=np.argsort(np.abs(rho))[::-1][:k]
    return list(zip(idx,rho[idx]))

print('Top-10 |rho| ARAT (Adv):',topk(sA_adv,10))
print('Top-10 |rho| FMA  (Adv):',topk(sF_adv,10))

fig,ax=plt.subplots(figsize=(12,4))
ax.plot(sA_adv,label='ARAT rho (adv)',alpha=0.8)
ax.plot(sF_adv,label='FMA rho (adv)',alpha=0.8)
ax.axvline(64,color='gray',ls='--',lw=1); ax.axvline(128,color='gray',ls='--',lw=1)
ax.set_title('Spearman profile by embedding dimension (Adv)')
ax.legend(); plt.tight_layout(); plt.savefig('results/figures/spearman_profile.png',dpi=150); plt.show()


## 9) ICC Reliability (Even vs Odd)


In [ ]:
def icc_3_1(x,y):
    n=len(x)
    m=(x+y)/2.0
    SS_between=2*n*m.var()
    SS_within=((x-m)**2 + (y-m)**2).sum()
    MS_between=SS_between/(n-1)
    MS_within=SS_within/n
    return (MS_between-MS_within)/(MS_between+MS_within)

def icc_profile(Xe,Xo):
    out=np.zeros(Xe.shape[1])
    for d in range(Xe.shape[1]):
        out[d]=icc_3_1(Xe[:,d],Xo[:,d])
    return out

icc_adv=icc_profile(adv['Xe'],adv['Xo'])
icc_std=icc_profile(std['Xe'],std['Xo'])

pd.DataFrame({'dim':np.arange(1028),'icc':icc_adv}).to_csv('results/metrics/icc_adv.csv',index=False)
pd.DataFrame({'dim':np.arange(1028),'icc':icc_std}).to_csv('results/metrics/icc_standard.csv',index=False)

print('Adv ICC >0.75:',int((icc_adv>0.75).sum()),'/1028')
print('Adv ICC 0.5-0.75:',int(((icc_adv>0.5)&(icc_adv<=0.75)).sum()),'/1028')
print('Adv ICC <=0.5:',int((icc_adv<=0.5).sum()),'/1028')

fig,ax=plt.subplots(figsize=(8,4))
sns.histplot(icc_adv,bins=40,ax=ax,color='tab:blue',alpha=0.7,label='Adv')
sns.histplot(icc_std,bins=40,ax=ax,color='tab:orange',alpha=0.5,label='Standard')
ax.legend(); ax.set_title('ICC distribution (3,1)');
plt.tight_layout(); plt.savefig('results/figures/icc_distribution.png',dpi=150); plt.show()


## 10) Summary + Full Metrics Export


In [ ]:
summary_rows=[
    {'section':'dataset','metric':'subjects_total','value':int(clin_df.subject.nunique())},
    {'section':'dataset','metric':'visits_total','value':int(len(clin_df))},
    {'section':'dataset','metric':'healthy_visits','value':int(((clin_df.ARAT==57)&(clin_df.FMA==66)).sum())},
    {'section':'regression','metric':'ARAT_R2_standard','value':res_std_arat['r2']},
    {'section':'regression','metric':'ARAT_R2_adv','value':res_adv_arat['r2']},
    {'section':'regression','metric':'FMA_R2_standard','value':res_std_fma['r2']},
    {'section':'regression','metric':'FMA_R2_adv','value':res_adv_fma['r2']},
    {'section':'regression','metric':'ARAT_spearman_standard','value':res_std_arat['spearman_r']},
    {'section':'regression','metric':'ARAT_spearman_adv','value':res_adv_arat['spearman_r']},
    {'section':'regression','metric':'FMA_spearman_standard','value':res_std_fma['spearman_r']},
    {'section':'regression','metric':'FMA_spearman_adv','value':res_adv_fma['spearman_r']},
    {'section':'icc_adv','metric':'count_gt_0_75','value':int((icc_adv>0.75).sum())},
    {'section':'icc_adv','metric':'count_gt_0_50','value':int((icc_adv>0.50).sum())},
]
summary_df=pd.DataFrame(summary_rows)
summary_df.to_csv('results/metrics/full_summary.csv',index=False)
display(summary_df)
